# Common — shared imports + bootstrap + schemas

`%run common.ipynb` from every agent / pipeline notebook. One source of truth for the
shared imports, the OpenRouter client, `model()` / `model_settings()`, `TODAY`, and the
cross-agent schemas (`TalkingPoint`, `TopicTheme`, `PlannerOutput`, `BrainstormOutput`).
Note: it also carries a few imports only `research_workflow` needs (httpx, bs4,
function_tool), so this is shared setup, not strictly per-agent bootstrap.

In [ ]:
# Common: shared imports + bootstrap + cross-agent schemas.
# %run by every agent / pipeline notebook. No research code here
# (that lives in research_workflow.ipynb).
import os
# Run from backend/ so .env and `%run common.ipynb` resolve regardless of the
# kernel's launch dir (IDE/Jupyter kernels often start at the workspace root).
if not os.path.exists("common.ipynb"):
    for _p in ("backend", "extras/socialApp/backend"):
        if os.path.exists(f"{_p}/common.ipynb"):
            os.chdir(_p)
            break

import os
import json
import asyncio
import time
import datetime
import re

import httpx
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel, Field
from agents import (
    Agent,
    Runner,
    OpenAIChatCompletionsModel,
    ModelSettings,
    function_tool,
    set_tracing_export_api_key,
)

load_dotenv(override=True)

# --- OpenRouter client ---
openrouter_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

def model(slug: str) -> OpenAIChatCompletionsModel:
    return OpenAIChatCompletionsModel(model=slug, openai_client=openrouter_client)

def model_settings(effort: str | None) -> ModelSettings:
    if not effort or effort.lower() in {"none", ""}:
        return ModelSettings()
    return ModelSettings(extra_body={"reasoning": {"effort": effort}})

RESEARCH_EFFORT   = "none"
BRAINSTORM_EFFORT = "none"
DRAFT_EFFORT      = "none"
TODAY = datetime.date.today().strftime("%Y-%m-%d")

if os.getenv("OPENAI_API_KEY"):
    set_tracing_export_api_key(os.environ["OPENAI_API_KEY"])

import nest_asyncio
nest_asyncio.apply()  # reentrant loop so `%run notebook.ipynb` works even when a
                      # dependency notebook has top-level-await cells

print(f"common ready. cwd={os.path.basename(os.getcwd())} TODAY={TODAY}")


In [ ]:
# Cell 2 — Schemas: TalkingPoint / TopicTheme / PlannerOutput / BrainstormOutput
# + render_brainstorm (the handoff renderer: BrainstormOutput -> Draft input block)

class TalkingPoint(BaseModel):
    angle: str = Field(
        description="A compact conversation angle, meaning a topic direction the user can build "
        "from. It should sound like a bullet-point idea in a prep note, not like a "
        "direct question to say out loud."
    )
    context: str | None = Field(
        default=None,
        description=(
            "Optional short background, 1-3 sentences. It should provide the user with the necessary information to understand the angle. "
            "Context can be skipped if the angle stands on its own."
        )
    )
    source_url: str | None = Field(
        default=None,
        description="Source URL if this point came from a research call. Copy the URL "
        "verbatim from the matching Source.url in the research output. Omit for "
        "first-principles points.",
    )


class TopicTheme(BaseModel):
    name: str = Field(
        description="Short cluster name, 1-2 words. Emerges from the points themselves — "
        "e.g. 'Food', 'Attractions', 'Culture', 'Recent News', 'Other'."
    )
    points: list[TalkingPoint]


class PlannerOutput(BaseModel):
    subject: str
    framing: str = Field(
        description="2-3 sentences that mix background knowledge + relevant recent context. "
        "This is what introduces the subject in the final card."
    )
    themes: list[TopicTheme] = Field(
        description="2-5 themed clusters of talking points."
    )
    reasoning_notes: str = Field(
        description="One short paragraph: which brainstorm angles were carried, plus a "
        "one-liner per research call about what it looked up and why."
    )


class BrainstormOutput(BaseModel):
    subject_facet_angles: list[TalkingPoint] = Field(
        description="Direction (a): angles about the subject itself — experiences, "
        "preferences, opinions, stories the person probably has. source_url must be null."
    )
    person_angles: list[TalkingPoint] = Field(
        description="Direction (b): angles about the person behind the phrase — goals, "
        "motivations, identity, origin story, plans. Never empty. source_url must be null."
    )
    research_gaps: list[str] = Field(
        description="1 to 3 focused research questions for facts that are time-sensitive or "
        "unknowable from first principles. At least one about recent developments in the "
        "subject. At most 3 — the next stage can research only 3 gaps."
    )


def render_brainstorm(bs: BrainstormOutput) -> str:
    """Render BrainstormOutput into the fixed-input block of the Draft agent's message."""
    lines = ["Brainstorm (fixed input):", "Subject facet angles:"]
    for pt in bs.subject_facet_angles:
        lines.append(f"- {pt.angle}" + (f"  (context: {pt.context})" if pt.context else ""))
    lines.append("Person angles:")
    for pt in bs.person_angles:
        lines.append(f"- {pt.angle}" + (f"  (context: {pt.context})" if pt.context else ""))
    lines.append("Research gaps:")
    for gap in bs.research_gaps:
        lines.append(f"- {gap}")
    return "\n".join(lines)


print("TalkingPoint, TopicTheme, PlannerOutput, BrainstormOutput, render_brainstorm defined.")
